# CS 3892 / 5892 — Session 14 · Equivalence, simulation and bisimulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-13-equivalence-simulation-and-bisimulation.ipynb)

**Tuesday, October 13, 2026.** When are two systems *the same*? Three answers, each finer than the
last: **trace equivalence** (the same runs, as an observer sees them), **simulation** (one can
match every move of the other) and **bisimulation** (each can match the other, with one relation).
The answer decides which properties carry over from one system to the other — LTL for the first,
all of CTL\* for the last — and it is what makes **abstraction** sound.

Every Python file asserts its own results. NuSMV (free, LGPL) is fetched by the setup cell.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab) and define helpers ---------------
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-13-equivalence-simulation-and-bisimulation"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

SMV = ROOT / "sessions" / SESSION / "smv"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    show(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. (Same cell as sessions 9-13.)
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older
# Colab image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import shutil

BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "01_vending_late_choice.smv")],
                           capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path):
    """Run NuSMV on a file in batch mode; return its output minus the banner."""
    r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))

def results(out):
    """Split NuSMV output into {printed formula: (verdict, counterexample text)}.
    NuSMV does not print verdicts in file order, so match them by formula."""
    res, key = {}, None
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            key = re.sub(r"\s+", " ", m.group(1)).strip()
            res[key] = [m.group(2), ""]
        elif key and res[key][0] == "false":
            res[key][1] += line + "\n"
    return {k: tuple(v) for k, v in res.items()}

def verdicts(path):
    """Print one line per property, and return {formula: verdict}."""
    out = smv(path)
    res = results(out)
    for k, (v, _) in res.items():
        print(f"{v:6} {k}")
    return {k: v for k, (v, _) in res.items()}

print("ready -- smv(path), results(out), verdicts(path)")

## 1. Same traces, different branching

Two vending machines. **LATE** takes the coin and *then* lets you choose. **EARLY** commits to a
drink as it takes the coin; both of its paid states show `paid` to an observer.
Every run of either reads `idle paid (coffee | tea) idle paid …`, so they have the **same traces**.

In [ ]:
from IPython.display import SVG, display
FIG = ROOT / "sessions" / SESSION / "figures"
display(SVG(filename=str(FIG / "vending_late.svg")))
display(SVG(filename=str(FIG / "vending_early.svg")))

**LTL cannot tell them apart; CTL can.** The four `LTLSPEC`s get the same verdicts in both files.
The two CTL `SPEC`s ask what is still *possible* at `paid`: true for LATE, false for EARLY.

In [ ]:
late, early = SMV / "01_vending_late_choice.smv", SMV / "02_vending_early_choice.smv"
show(late); show(early)
L = [v for v, _ in results(smv(late)).values()]
E = [v for v, _ in results(smv(early)).values()]
print("            CTL          LTL")
print("LATE :", L[:2], L[2:])
print("EARLY:", E[:2], E[2:])
assert L == ["true", "true", "true", "true", "false", "false"]
assert E == ["false", "false", "true", "true", "false", "false"]
assert L[2:] == E[2:] and L[:2] != E[:2]
print("\nLTL verdicts agree; the CTL verdicts do not")

## 2. Trace equivalence, simulation, bisimulation

`03_three_equivalences.py` computes all three. LATE **simulates** EARLY (it can match every move),
EARLY cannot simulate LATE, and so they are not bisimilar. The second pair, FAIR and JAMS,
**simulate each other** and are *still* not bisimilar: simulation in both directions is weaker
than bisimulation.

In [ ]:
display(SVG(filename=str(FIG / "fair_jams.svg")))
out = run(PYD / "03_three_equivalences.py")
assert "bisimilar => simulate each other => same traces" in out

## 3. Computing bisimilarity: partition refinement

Start with one block per label; split any block whose states disagree about which blocks they can
move to; repeat until stable. This is the algorithm that minimized DFAs in session 13.
END_D3 collapses to the two-state END_D; the vending machines' initial states are pulled apart.

In [ ]:
display(SVG(filename=str(FIG / "ends_d3.svg")))
out = run(PYD / "04_partition_refinement.py")
assert "3 states were 2 all along" in out and "NOT bisimilar" in out

## 4. Why simulation matters: abstraction

Map the agent's five states to two — is a delete happening, or not. The abstraction **simulates**
the agent, so an LTL property that holds on the abstraction holds on the agent. A property that
*fails* on the abstraction may fail only because the abstraction is too coarse: a **spurious**
counterexample, and the cue to refine.

In [ ]:
display(SVG(filename=str(FIG / "agent_abstraction.svg")))
out = run(PYD / "05_abstraction_simulates.py")
assert "SPURIOUS" in out

In [ ]:
# NuSMV on the same two properties: the abstraction (06), then the agent itself (07).
A = SMV / "06_agent_abstract.smv"; C = SMV / "07_agent_concrete.smv"
show(A); show(C)
ra, rc = results(smv(A)), results(smv(C))
va, vc = [v for v, _ in ra.values()], [v for v, _ in rc.values()]
print("abstraction:", va, "   agent:", vc)
assert va == ["true", "false"] and vc == ["true", "true"]
print("\nthe counterexample NuSMV finds on the abstraction -- del, quiet, del:")
print(list(ra.values())[1][1])

## Try it yourself

- In `03`, add a third vending machine that sometimes keeps the coin and returns to `idle`.
  Is it trace equivalent to LATE? Does either simulate the other?
- In `04`, add a fourth state to END_D3 that is a copy of `q1`. How many classes are left?
- In `05`, find a property that holds on the agent but fails on **both** abstractions, and the
  refinement that fixes it.